In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date
from pyspark.sql.types import DecimalType

In [0]:
bronze_df = spark.read.table("workspace.default.bronze_transactions")

In [0]:
# Apply transformations: cast types, add processing timestam
silver_df=bronze_df.withColumn("amount",col("amount").cast(DecimalType(10,2))).withColumn("transaction_date",to_date(col("transaction_date"),"yyyy-MM-dd")).withColumn("processed_at",current_timestamp())

In [0]:
display(silver_df)

In [0]:
# Filter out invalid records: null transaction_id or negative amounts
silver_df = silver_df.filter(
    (col("transaction_id").isNotNull()) & (col("amount") >= 0)
)

In [0]:
silver_df.display()

In [0]:
from delta.tables import DeltaTable

In [0]:
# Table name for the silver layer
silver_table_name = "workspace.default.silver_transactions"

In [0]:
# Check if the silver table already exists
if not spark.catalog.tableExists(silver_table_name):
    # First run: create the silver table from the transformed DataFrame
    silver_df.write.format("delta").saveAsTable(silver_table_name)
else:
    # Subsequent runs: upsert using MERGE on transaction_id
    delta_table = DeltaTable.forName(spark, silver_table_name)
    
    # Deduplicate source to ensure one row per transaction_id
    deduplicated_df = silver_df.dropDuplicates(["transaction_id"])

    (delta_table.alias("target")
        .merge(deduplicated_df.alias("source"), "target.transaction_id = source.transaction_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .withSchemaEvolution()
        .execute()
    )

print("Silver table upsert complete.")

In [0]:
# Verify the silver table: check schema and row contents
result = spark.sql("SELECT * FROM workspace.default.silver_transactions")
result.display()

# Print the schema to confirm data types
result.printSchema()

In [0]:
# Check the silver table schema to confirm the new 'category' column exists
spark.sql("DESCRIBE workspace.default.silver_transactions").display()

In [0]:
# Verify old records have null category, new records have category populated
spark.sql("""
    SELECT transaction_id, customer_id, merchant, category
    FROM workspace.default.silver_transactions
    ORDER BY transaction_id
""").display()

In [0]:
# -----------------------------------------------
# Data quality checks on the silver table
# -----------------------------------------------
silver_df = spark.read.table("workspace.default.silver_transactions")


In [0]:
# Check 1: No null transaction_ids (primary key integrity)
null_count = silver_df.filter("transaction_id IS NULL").count()
assert null_count == 0, f"Quality check failed: {null_count} null transaction_ids found"

In [0]:
# Check 2: All amounts within valid range (0 to 1,000,000)
invalid_amounts = silver_df.filter("amount < 0 OR amount > 1000000").count()
assert invalid_amounts == 0, f"Quality check failed: {invalid_amounts} amounts out of valid range"


In [0]:
# Check 3: No duplicate transaction_ids (uniqueness check)
total_count = silver_df.count()
distinct_count = silver_df.select("transaction_id").distinct().count()
assert total_count == distinct_count, f"Quality check failed: {total_count - distinct_count} duplicate transaction_ids found"

In [0]:
print("All quality checks passed!")